In [1]:
## imports
import pandas as pd
import numpy as np
import re
import requests
import yaml


## repeated printouts
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# 1. Example 1: no credentials; no wrapper

Site: National Assessment of Education Progress (NAEP)

Documentation: https://www.nationsreportcard.gov/api_documentation.aspx

Base link: https://www.nationsreportcard.gov/DataService/GetAdhocData.aspx 

## 1.1 Query to pull some data

In [2]:
## using their example query of 2011 writing scores separated by gender
## based on here - https://stackoverflow.com/questions/40836749/pythonic-way-of-writing-a-single-line-long-string
## using the ( ) syntax to formulate a long
## string without linebreaks added
example_naep_query = (
'https://www.nationsreportcard.gov/'
'Dataservice/GetAdhocData.aspx?'
'type=data&subject=writing&grade=8&'
'subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2011')


example_naep_query


'https://www.nationsreportcard.gov/Dataservice/GetAdhocData.aspx?type=data&subject=writing&grade=8&subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2011'

In [8]:
## use requests to call the api
naep_resp = requests.get(example_naep_query)
naep_resp
print(type(naep_resp))

## get the json contents of the response 
## here, we're assuming valid response
naep_resp_j = naep_resp.json()
naep_resp_j

## with result, turn it into a dataframe
naep_resp_d = pd.DataFrame(naep_resp_j['result'])
naep_resp_d

<Response [200]>

<class 'requests.models.Response'>


{'status': 200,
 'serviceVersion': '6.4.2026.1',
 'dwellTimeMS': '15.6467',
 'avgWebHostCPUTotalLoad': 'N/A',
 'dataHitType': 'FROM_MEMORY',
 'Source': 'B11A',
 'result': [{'year': 2011,
   'sample': 'R3',
   'yearSampleLabel': '2011',
   'Cohort': 2,
   'CohortLabel': 'Grade 8',
   'stattype': 'MN:MN',
   'subject': 'WRI',
   'grade': 8,
   'scale': 'WRIRP',
   'jurisdiction': 'NP',
   'jurisLabel': 'National public',
   'variable': 'GENDER',
   'variableLabel': 'Sex',
   'varValue': '1',
   'varValueLabel': 'Male',
   'value': 139.099504632971,
   'isStatDisplayable': 1,
   'errorFlag': 0},
  {'year': 2011,
   'sample': 'R3',
   'yearSampleLabel': '2011',
   'Cohort': 2,
   'CohortLabel': 'Grade 8',
   'stattype': 'MN:MN',
   'subject': 'WRI',
   'grade': 8,
   'scale': 'WRIRP',
   'jurisdiction': 'NP',
   'jurisLabel': 'National public',
   'variable': 'GENDER',
   'variableLabel': 'Sex',
   'varValue': '2',
   'varValueLabel': 'Female',
   'value': 158.567104984955,
   'isStatDispl

,year,sample,yearSampleLabel,Cohort,CohortLabel,stattype,subject,grade,scale,jurisdiction,jurisLabel,variable,variableLabel,varValue,varValueLabel,value,isStatDisplayable,errorFlag
0,2011,R3,2011,2,Grade 8,MN:MN,WRI,8,WRIRP,NP,National public,GENDER,Sex,1,Male,139.099505,1,0
1,2011,R3,2011,2,Grade 8,MN:MN,WRI,8,WRIRP,NP,National public,GENDER,Sex,2,Female,158.567105,1,0


## 1.2 What happens if there's an error in our query?

In [4]:
## here's a query that from the documentation we know
## won't work since i modified year to 2025 which doesnt
## exist in the data
wrong_naep_query = (
'https://www.nationsreportcard.gov/'
'Dataservice/GetAdhocData.aspx?'
'type=data&subject=writing&grade=8&'
'subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2025')

wrong_naep_query

'https://www.nationsreportcard.gov/Dataservice/GetAdhocData.aspx?type=data&subject=writing&grade=8&subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2025'

In [28]:
## use requests to call the api
naep_wrong_resp = requests.get(wrong_naep_query)
naep_wrong_resp

<Response [400]>

In [ ]:
## in the case of this particular api,
## the call returns some response but
## when we try to extract the json containing
## status or results, we get in an error
#naep_wrong_resp.json() # uncomment to see error

### 1.2.2 More all-purpose way of allowing remainder of calls to run: try, except

In [6]:
## putting it in a try; except as general error catching
try:
    results = naep_wrong_resp.json()['result']
except Exception as e:
    print('Failed to get result from API due to error:')
    print(e) # or just: pass

Failed to get result from API due to error:
Invalid control character at: line 1 column 293 (char 292)


### 1.2.3 Can usually also find more targeted way but that varies more across APIs

In [7]:
## if we wanted do more specific error catching,
## see that the status == 400 actually appears here
## so could write if else along those lines
naep_wrong_resp.text
naep_resp.text

if "System.Exception" in naep_wrong_resp.text:
    print("NAEP results not found")

'{"statusCode":400,"result": "System.Exception: The query \'SELECT DISTINCT Framework FROM Cycles WHERE Subject=\'WRI\' AND Cohort=2 AND CONVERT(VARCHAR(10),Year)+Sample IN (\'2025R3\')\' did not return exactly 1 framework. Make sure you can trend the years defined for the given subject and cohort.\r\n   at NRCDataService3.GetAdhocData.GetFramework(NDEContext& ndeContext, String subjectCode, List`1 yearSamples, String cohort) in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 2733\r\n   at NRCDataService3.GetAdhocData.PopulateBaseOrchestratorRequest() in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 2348\r\n   at NRCDataService3.GetAdhocData.ConstructRequest_Datapoint() in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 947\r\n   at NRCDataService3.GetAdhocData.Page_Load(Object sender, EventArgs e) in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 354"}'

'{"status":200,"serviceVersion":"6.4.2026.1","dwellTimeMS":"19.6179","avgWebHostCPUTotalLoad":"N/A","dataHitType":"FROM_MEMORY","Source":"B11A","result": [{"year":2011,"sample":"R3","yearSampleLabel":"2011","Cohort":2,"CohortLabel":"Grade 8","stattype":"MN:MN","subject":"WRI","grade":8,"scale":"WRIRP","jurisdiction":"NP","jurisLabel":"National public","variable":"GENDER","variableLabel":"Sex","varValue":"1","varValueLabel":"Male","value":139.099504632971,"isStatDisplayable":1,"errorFlag":0},{"year":2011,"sample":"R3","yearSampleLabel":"2011","Cohort":2,"CohortLabel":"Grade 8","stattype":"MN:MN","subject":"WRI","grade":8,"scale":"WRIRP","jurisdiction":"NP","jurisLabel":"National public","variable":"GENDER","variableLabel":"Sex","varValue":"2","varValueLabel":"Female","value":158.567104984955,"isStatDisplayable":1,"errorFlag":0}]}'

NAEP results not found


## Activity 1: writing a function to make multiple, sequential calls

- Say we want to pull the data for grades 4, 8, and 12
- How can we write a function that iterates over a list of those grades and pulls the data for each grade?

**Note**: an ideal function would have arguments for each parameter in the API like subject, subscale, etc. Here we can leave those other parts constant

In [42]:
def apithing(grades):
    dataframes = []

    for grade in grades:
        naep_query = (
            "https://www.nationsreportcard.gov/"
            "Dataservice/GetAdhocData.aspx?"
            f"type=data&subject=writing&grade={grade}&"
            "subscale=WRIRP&variable=GENDER&"
            "jurisdiction=NP&stattype=MN:MN&Year=2011"
        )

        try:
            naep_resp = requests.get(naep_query)
            naep_resp.raise_for_status()

            naep_j = naep_resp.json()

            # Only use the contents under "result"
            grade_df = pd.DataFrame(naep_j["result"])

            dataframes.append(grade_df)

        except Exception as e:
            print(f"Failed for grade {grade}:")
            print(e)

    if len(dataframes) == 0:
        return pd.DataFrame()

    return pd.concat(dataframes, ignore_index=True)


results = apithing([4, 8, 12])
results

Failed for grade 4:
400 Client Error: Bad Request for url: https://www.nationsreportcard.gov/Dataservice/GetAdhocData.aspx?type=data&subject=writing&grade=4&subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2011


,year,sample,yearSampleLabel,Cohort,CohortLabel,stattype,subject,grade,scale,jurisdiction,jurisLabel,variable,variableLabel,varValue,varValueLabel,value,isStatDisplayable,errorFlag
0,2011,R3,2011,2,Grade 8,MN:MN,WRI,8,WRIRP,NP,National public,GENDER,Sex,1,Male,139.099505,1,0
1,2011,R3,2011,2,Grade 8,MN:MN,WRI,8,WRIRP,NP,National public,GENDER,Sex,2,Female,158.567105,1,0
2,2011,R3,2011,3,Grade 12,MN:MN,WRI,12,WRIRP,NP,National public,GENDER,Sex,1,Male,141.256978,1,0
3,2011,R3,2011,3,Grade 12,MN:MN,WRI,12,WRIRP,NP,National public,GENDER,Sex,2,Female,155.385917,1,0


# 2. Example 2: needs credentials; no wrapper

Create an account here: https://www.yelp.com/developers/v3/manage_app

In [43]:
## get the key
API_KEY = "yt1DhXGmclnqxjjraQaL2uXEDcSpRwm5A3oVf7QkY6G08LZZiqdDuHgMpzVajOkRioPzJa4vXA3YxVeM60H0FOCQcoIoucOHvAvktcTJGUMcrsQt6OJgU0I0fqV0anYx"

In [44]:
## use documentation to define what to search
## doc: https://www.yelp.com/developers/documentation/v3/business_search
## write the query 
base_url = "https://api.yelp.com/v3/businesses/search?"
my_name = "restaurants"
my_location = "Hanover,NH,03755"
yelp_genquery = ('{base_url}'
                'term={name}'
                '&location={loc}').format(base_url = base_url,
                name = my_name,
                loc = my_location)

## use requests to call the API; here, we're
## passing it our credentials (structure varies
## by API and telling it to only return 10 results
## (max is 50 at once)
header = {'Authorization': f"Bearer {API_KEY}"}
yelp_genresp = requests.get(yelp_genquery, headers = header)
yelp_genresp

## then, look at structure of response
yelp_genjson = yelp_genresp.json()


<Response [200]>

In [45]:
## example business
yelp_genjson['businesses'][0]

## more automatic way of summarizing but things end up in lists
## within columns for things like categories
yelp_gendf = pd.DataFrame(yelp_genjson['businesses'])
yelp_gendf.head()

{'id': '8ybF6YyRldtZmU9jil4xlg',
 'alias': 'mollys-restaurant-and-bar-hanover',
 'name': "Molly's Restaurant & Bar",
 'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/TJLrrA6z-SnPgZfrs2GQNQ/o.jpg',
 'is_closed': False,
 'url': 'https://www.yelp.com/biz/mollys-restaurant-and-bar-hanover?adjust_creative=lXpYO8jEsbLfOUHvIhlWEg&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=lXpYO8jEsbLfOUHvIhlWEg',
 'review_count': 582,
 'categories': [{'alias': 'tradamerican', 'title': 'American'},
  {'alias': 'burgers', 'title': 'Burgers'},
  {'alias': 'pizza', 'title': 'Pizza'}],
 'rating': 3.9,
 'coordinates': {'latitude': 43.701144, 'longitude': -72.2894249},
 'transactions': ['delivery'],
 'price': '$$',
 'location': {'address1': '43 South Main St',
  'address2': '',
  'address3': '',
  'city': 'Hanover',
  'zip_code': '03755',
  'country': 'US',
  'state': 'NH',
  'display_address': ['43 South Main St', 'Hanover, NH 03755']},
 'phone': '+16036432570',
 'display_phone': '(

,id,alias,name,image_url,is_closed,url,review_count,categories,rating,coordinates,transactions,price,location,phone,display_phone,distance,business_hours,attributes
0,8ybF6YyRldtZmU9jil4xlg,mollys-restaurant-and-bar-hanover,Molly's Restaurant & Bar,https://s3-media0.fl.yelpcdn.com/bphoto/TJLrrA...,False,https://www.yelp.com/biz/mollys-restaurant-and...,582,"[{'alias': 'tradamerican', 'title': 'American'...",3.9,"{'latitude': 43.701144, 'longitude': -72.2894249}",[delivery],$$,"{'address1': '43 South Main St', 'address2': '...",+16036432570,(603) 643-2570,250.830160,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'http://www.mollysrestaurant.com/...
1,XVGEEIH5rVB2QzW-qywcJw,base-camp-cafe-hanover,Base Camp Cafe,https://s3-media0.fl.yelpcdn.com/bphoto/tScZeo...,False,https://www.yelp.com/biz/base-camp-cafe-hanove...,264,"[{'alias': 'himalayan', 'title': 'Himalayan/Ne...",4.4,"{'latitude': 43.700626, 'longitude': -72.2887803}",[delivery],$$,"{'address1': '3 Lebanon St', 'address2': 'Ste ...",+16036432007,(603) 643-2007,196.139758,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'http://basecampcafenewhampshire....
2,JFE0XffhpP3Bi3DQRvymVg,little-havana-hanover,Little Havana,https://s3-media0.fl.yelpcdn.com/bphoto/941vaR...,False,https://www.yelp.com/biz/little-havana-hanover...,29,"[{'alias': 'cuban', 'title': 'Cuban'}, {'alias...",4.9,"{'latitude': 43.700743, 'longitude': -72.287599}","[delivery, pickup]",NaN,"{'address1': '15 Lebanon St', 'address2': '', ...",+18383831000,(838) 383-1000,102.833229,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'https://www.canva.com/design/DAG...
3,wyV_NfYn4ZOfp_sHMDPcAw,bistro-at-six-hanover,Bistro at Six,https://s3-media0.fl.yelpcdn.com/bphoto/i4jvss...,False,https://www.yelp.com/biz/bistro-at-six-hanover...,2,"[{'alias': 'lounges', 'title': 'Lounges'}, {'a...",4.0,"{'latitude': 43.7001146, 'longitude': -72.2877...",[],$$,"{'address1': '6 E South St', 'address2': '', '...",+16036430600,(603) 643-0600,198.651788,"[{'open': [{'is_overnight': True, 'start': '00...",{}
4,5WW4g_LRwau29KyjZGLyAA,sawtooth-kitchen-hanover,Sawtooth Kitchen,https://s3-media0.fl.yelpcdn.com/bphoto/61MNG4...,False,https://www.yelp.com/biz/sawtooth-kitchen-hano...,39,"[{'alias': 'chickenshop', 'title': 'Chicken Sh...",4.2,"{'latitude': 43.70158, 'longitude': -72.289641}",[],NaN,"{'address1': '33 S Main St', 'address2': '', '...",+16036435134,(603) 643-5134,242.607552,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'https://www.sawtoothkitchen.com/...


In [ ]:
## more data-specific way of summarizing
## we're doing a simple approach and just retaining
## cols that have a simple str structure
## if doing for real, would want to extract things
def clean_yelp_json(one_biz):

    ## restrict to str cols
    d_str = {key:value for key, value in one_biz.items()
             if type(value) == str}
    
    df_str = pd.DataFrame(d_str, index = [d_str['id']])
    return(df_str)

yelp_stronly = [clean_yelp_json(one_b) for one_b in yelp_genjson['businesses']]
yelp_stronly_df = pd.concat(yelp_stronly)

yelp_stronly_df.head(7)


# Activity 2: pull restaurants in a different location

- Try running a business search query for your hometown or another place by constructing a query similar to `yelp_genquery` but changing the location parameter
- Other endpoints require feeding what's called the business' fusion id into the API. Take an id from `yelp_stronly.id` and use the documentation here to pull the reviews for that business: https://docs.developer.yelp.com/reference/v3_business_reviews
- **Challenge**: generalize the previous step by writing a function that (1) takes a list of business ids as an input, (2) calls the reviews API for each id, (3) returns the results, and (4) rowbinds all results, i.e. turns them into a single, usable DataFrame

In [51]:
## use documentation to define what to search
## doc: https://www.yelp.com/developers/documentation/v3/business_search
## write the query 
base_url = "https://api.yelp.com/v3/businesses/search?"
my_name = "restaurants"
my_location = "Los Angeles,CA,90049"
yelp_genquery = ('{base_url}'
                'term={name}'
                '&location={loc}').format(base_url = base_url,
                name = my_name,
                loc = my_location)

## use requests to call the API; here, we're
## passing it our credentials (structure varies
## by API and telling it to only return 10 results
## (max is 50 at once)
header = {'Authorization': f"Bearer {API_KEY}"}
yelp_genresp = requests.get(yelp_genquery, headers = header)
yelp_genresp

## then, look at structure of response
yelp_genjson = yelp_genresp.json()

## example business
yelp_genjson['businesses'][0]

## more automatic way of summarizing but things end up in lists
## within columns for things like categories
yelp_gendf = pd.DataFrame(yelp_genjson['businesses'])
yelp_gendf

<Response [200]>

{'id': 'kSU0vkplwy_q5km2oGz6Hg',
 'alias': 'neighborly-los-angeles',
 'name': 'Neighborly',
 'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/A_MNZntOPY9ZU2ceZpHpLQ/o.jpg',
 'is_closed': False,
 'url': 'https://www.yelp.com/biz/neighborly-los-angeles?adjust_creative=lXpYO8jEsbLfOUHvIhlWEg&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=lXpYO8jEsbLfOUHvIhlWEg',
 'review_count': 107,
 'categories': [{'alias': 'mediterranean', 'title': 'Mediterranean'},
  {'alias': 'tradamerican', 'title': 'American'},
  {'alias': 'pizza', 'title': 'Pizza'}],
 'rating': 4.5,
 'coordinates': {'latitude': 34.052954866891696, 'longitude': -118.4677141},
 'transactions': ['delivery', 'pickup'],
 'location': {'address1': '11754 San Vicente Blvd',
  'address2': '',
  'address3': '',
  'city': 'Los Angeles',
  'zip_code': '90049',
  'country': 'US',
  'state': 'CA',
  'display_address': ['11754 San Vicente Blvd', 'Los Angeles, CA 90049']},
 'phone': '+14242894124',
 'display_phone': '(

,id,alias,name,image_url,is_closed,url,review_count,categories,rating,coordinates,transactions,location,phone,display_phone,distance,business_hours,attributes,price
0,kSU0vkplwy_q5km2oGz6Hg,neighborly-los-angeles,Neighborly,https://s3-media0.fl.yelpcdn.com/bphoto/A_MNZn...,False,https://www.yelp.com/biz/neighborly-los-angele...,107,"[{'alias': 'mediterranean', 'title': 'Mediterr...",4.5,"{'latitude': 34.052954866891696, 'longitude': ...","[delivery, pickup]","{'address1': '11754 San Vicente Blvd', 'addres...",+14242894124,(424) 289-4124,4040.475520,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'http://beneighborly.com/'},NaN
1,IQYl-iQBkSqyR562eFxlbg,perse-restaurant-los-angeles,Perse Restaurant,https://s3-media0.fl.yelpcdn.com/bphoto/Jr_mRg...,False,https://www.yelp.com/biz/perse-restaurant-los-...,81,"[{'alias': 'persian', 'title': 'Persian/Irania...",4.3,"{'latitude': 34.05395959470716, 'longitude': -...","[delivery, pickup]","{'address1': '11677 San Vicente Blvd', 'addres...",+14242572010,(424) 257-2010,4034.387592,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'https://perserestaurant.com/menu/'},NaN
2,b30f4sBlo3y1_vUnFvYzTg,great-white-los-angeles-3,Great White,https://s3-media0.fl.yelpcdn.com/bphoto/-KzSom...,False,https://www.yelp.com/biz/great-white-los-angel...,103,"[{'alias': 'newamerican', 'title': 'New Americ...",4.1,"{'latitude': 34.05348, 'longitude': -118.46336}",[],"{'address1': '11660 Darlington Ave', 'address2...",+14243256644,(424) 325-6644,4182.696668,"[{'open': [{'is_overnight': False, 'start': '0...",{'menu_url': 'https://www.greatwhite.cafe/menus'},NaN
3,xmQggU4JUAN1fcGi8_kZXQ,terroni-los-angeles-2,Terroni,https://s3-media0.fl.yelpcdn.com/bphoto/JGT3nq...,False,https://www.yelp.com/biz/terroni-los-angeles-2...,18,"[{'alias': 'italian', 'title': 'Italian'}]",3.4,"{'latitude': 34.065542, 'longitude': -118.4691}",[],"{'address1': '155 S Barrington Pl', 'address2'...",+14242083254,(424) 208-3254,2745.775548,"[{'open': [{'is_overnight': False, 'start': '1...",{},NaN
4,ED2n1JOkKlJUeedj2ZrsCg,cafe-landwer-los-angeles-2,Cafe Landwer,https://s3-media0.fl.yelpcdn.com/bphoto/8ClJ7p...,False,https://www.yelp.com/biz/cafe-landwer-los-ange...,27,"[{'alias': 'mediterranean', 'title': 'Mediterr...",4.1,"{'latitude': 34.054142633866135, 'longitude': ...","[delivery, pickup]","{'address1': '11677 San Vicente Blvd', 'addres...",+14243715008,(424) 371-5008,4028.008076,"[{'open': [{'is_overnight': False, 'start': '0...",{'menu_url': 'https://landwercafe.com/wp-conte...,NaN
5,m3BwegCaXnMQVVZS1e3Nkg,sun-nong-dan-los-angeles-6,Sun Nong Dan,https://s3-media0.fl.yelpcdn.com/bphoto/uUmzkl...,False,https://www.yelp.com/biz/sun-nong-dan-los-ange...,935,"[{'alias': 'korean', 'title': 'Korean'}, {'ali...",4.8,"{'latitude': 34.0436787, 'longitude': -118.446...",[],"{'address1': '1803 Sawtelle Blvd', 'address2':...",+13107103988,(310) 710-3988,5945.047044,"[{'open': [{'is_overnight': True, 'start': '00...",{'menu_url': 'https://sunnongdanusa.com/menu-w...,$$
6,aWlNsaC0YYMVmw4oEWr91A,ggiata-los-angeles-16,Ggiata,https://s3-media0.fl.yelpcdn.com/bphoto/2riA6O...,False,https://www.yelp.com/biz/ggiata-los-angeles-16...,114,"[{'alias': 'sandwiches', 'title': 'Sandwiches'...",4.2,"{'latitude': 34.053457, 'longitude': -118.462999}",[],"{'address1': '11640 San Vicente Blvd', 'addres...",+14244844800,(424) 484-4800,4192.487904,"[{'open': [{'is_overnight': False, 'start': '0...",{},NaN
7,mZHRbM4Xc-KXTZ6O90m0Iw,cafe-belen-los-angeles-2,Cafe Belen,https://s3-media0.fl.yelpcdn.com/bphoto/jPTIQ7...,False,https://www.yelp.com/biz/cafe-belen-los-angele...,543,"[{'alias': 'breakfast_brunch', 'title': 'Break...",4.8,"{'latitude': 34.04186, 'longitude': -118.4612}","[delivery, pickup, restaurant_reservation]","{'address1': '11925 Santa Monica Blvd', 'addre...",+14242930193,(424) 293-0193,5409.139548,"[{'open': [{'is_overnight': False, 'start': '0...",{'menu_url': 'https://www.cafe-belen.com/menu'...,$$
8,mU90DGgIKNY183